# 03 — Da configuração para objetos tipados

O notebook [01](01_formatos_de_configuracao.ipynb) terminou com dicionários. Aqui eles viram **objetos do domínio**, por cinco
caminhos, do mais manual ao mais completo:

| caminho | converte tipos? | valida? | aninhamento | dependência |
|---|---|---|---|---|
| `Dataclass(**d)` | ❌ | ❌ | ❌ | nenhuma |
| conversor à mão (`get_type_hints`) | ✅ (o que você implementar) | parcial | ✅ | nenhuma |
| `TypeAdapter(Dataclass)` | ✅ | ✅ | ✅ | Pydantic (a classe continua stdlib) |
| `BaseModel` / `pydantic.dataclasses` | ✅ | ✅ | ✅ | Pydantic na classe |
| `BaseSettings` (pydantic-settings) | ✅ | ✅ | ✅ + **camadas** (arquivos, env, args) | pydantic-settings |

In [1]:
import os
from dataclasses import dataclass, field, fields, is_dataclass
from datetime import date
from enum import Enum, StrEnum
from pathlib import Path
from typing import Any, get_args, get_origin, get_type_hints

import yaml

CONFIGS = Path("configs")
raw = yaml.safe_load((CONFIGS / "base.yaml").read_text())
print(f"raw: {raw}")

raw: {'app': {'name': 'relatorios', 'environment': 'dev'}, 'server': {'host': '0.0.0.0', 'port': 8080, 'timeout_s': 2.5}, 'database': {'url': 'postgresql://localhost:5432/app', 'pool_size': 5}, 'log_level': 'info', 'features': ['export_csv']}


## 1. Dataclass pura: `**dict` não basta

In [2]:
class LogLevel(StrEnum):
    DEBUG = "debug"
    INFO = "info"
    WARNING = "warning"


class Environment(StrEnum):
    DEV = "dev"
    PROD = "prod"


@dataclass(frozen=True)
class AppSection:
    name: str
    environment: Environment


@dataclass(frozen=True)
class ServerSection:
    host: str
    port: int
    timeout_s: float = 5.0


@dataclass(frozen=True)
class DatabaseSection:
    url: str
    pool_size: int = 5


@dataclass(frozen=True)
class Config:
    __pydantic_config__ = {"extra": "forbid"}     # sem anotação: não é campo; explicado na seção 3
    app: AppSection
    server: ServerSection
    database: DatabaseSection
    log_level: LogLevel = LogLevel.INFO
    features: tuple[str, ...] = ()


naive = Config(**raw)
print(f"type(naive.server): {type(naive.server).__name__}")
print(f"type(naive.log_level): {type(naive.log_level).__name__}")
print(f"type(naive.features): {type(naive.features).__name__}")

type(naive.server): dict
type(naive.log_level): str
type(naive.features): list


O `@dataclass` não converte nada: `server` continua `dict`, `log_level` continua `str`, `features` continua `list` (apesar da
anotação `tuple`), e `frozen=True` não impede `naive.server["port"] = ...`. As anotações são **documentação**, não contrato.

## 2. Conversor à mão com `get_type_hints`

Para ver o que bibliotecas como `dacite`/`cattrs` fazem por dentro: percorrer as anotações e converter recursivamente.

In [3]:
def from_dict(cls: type, data: Any) -> Any:
    """Converte `data` para o tipo `cls`: dataclasses, Enums, tuple[...], list[...], date e escalares."""
    origin, args = get_origin(cls), get_args(cls)
    if is_dataclass(cls):
        if not isinstance(data, dict):
            raise TypeError(f"{cls.__name__}: esperava mapeamento, veio {type(data).__name__}")
        hints = get_type_hints(cls)
        known = {f.name for f in fields(cls)}
        unknown = set(data) - known
        if unknown:
            raise TypeError(f"{cls.__name__}: chaves desconhecidas {sorted(unknown)}")
        return cls(**{name: from_dict(hints[name], value) for name, value in data.items()})
    if isinstance(cls, type) and issubclass(cls, Enum):
        return cls(data)
    if origin in (tuple, list):
        return origin(from_dict(args[0], item) for item in data)
    if cls is date and isinstance(data, str):
        return date.fromisoformat(data)
    if cls in (int, float, str, bool):
        if cls is bool and not isinstance(data, bool):
            raise TypeError(f"esperava bool, veio {data!r}")
        return cls(data)
    return data


cfg = from_dict(Config, raw)
print(f"cfg.server: {cfg.server!r}")
print(f"cfg.log_level: {cfg.log_level!r}")
print(f"cfg.features: {cfg.features!r}")
for bad in [{**raw, "log_level": "verbose"}, {**raw, "server": {**raw["server"], "prot": 1}}]:
    try:
        from_dict(Config, bad)
    except (TypeError, ValueError) as ex:
        print(f"{type(ex).__name__}: {ex}")

cfg.server: ServerSection(host='0.0.0.0', port=8080, timeout_s=2.5)
cfg.log_level: <LogLevel.INFO: 'info'>
cfg.features: ('export_csv',)
ValueError: 'verbose' is not a valid LogLevel
TypeError: ServerSection: chaves desconhecidas ['prot']


Funciona para o caso feliz, mas olhe o que ficou de fora: restrições (`port` entre 1 e 65535), uniões (`str | None`), mensagens com
o **caminho** do erro (`server.prot`), coleta de **todos** os erros de uma vez, e `int("8080.7")`/`int(True)` passando. Cada item é
mais código para manter — é aí que entra o Pydantic.

## 3. `TypeAdapter` sobre a dataclass **stdlib**

O Pydantic valida dataclasses comuns sem mudar a classe. O domínio continua livre de dependências; a validação fica na fronteira.

In [4]:
from pydantic import ConfigDict, TypeAdapter, ValidationError

config_adapter = TypeAdapter(Config)
cfg = config_adapter.validate_python(raw)
print(f"cfg.server: {cfg.server!r}")
print(f"cfg.features: {cfg.features!r}")
print(f"type(cfg).__module__: {type(cfg).__module__}")       # continua a nossa dataclass

cfg.server: ServerSection(host='0.0.0.0', port=8080, timeout_s=2.5)
cfg.features: ('export_csv',)
type(cfg).__module__: __main__


> ⚠️ **Chaves extras:** validada pelo Pydantic, uma dataclass stdlib **ignora** chaves desconhecidas por padrão — erro de
> digitação vira default silencioso:

In [5]:
print(f"TypeAdapter(ServerSection) com 'prot': {TypeAdapter(ServerSection).validate_python({'host': 'x', 'port': 1, 'prot': 2})!r}")

TypeAdapter(ServerSection) com 'prot': ServerSection(host='x', port=1, timeout_s=5.0)


Por isso `Config` declara `__pydantic_config__ = {"extra": "forbid"}`: um atributo de classe (sem anotação, então não vira campo),
que é um `dict` comum — o domínio continua sem importar o Pydantic. A configuração vale também para as dataclasses stdlib
**aninhadas** (`server.prot` abaixo é rejeitado mesmo `ServerSection` não tendo a sua). O `TypeAdapter(..., config=...)` é
**recusado** para dataclasses, inclusive as stdlib.

Erros com caminho completo, todos de uma vez:

In [6]:
broken = {
    "app": {"name": "relatorios", "environment": "staging"},
    "server": {"host": "0.0.0.0", "port": "oitenta", "prot": 1},
    "database": {"url": "postgresql://x"},
    "log_level": "verbose",
}
try:
    config_adapter.validate_python(broken)
except ValidationError as ex:
    for error in ex.errors():
        print(f"{'.'.join(map(str, error['loc']))}: {error['msg']}")

app.environment: Input should be 'dev' or 'prod'
server.port: Input should be a valid integer, unable to parse string as an integer
server.prot: Unexpected keyword argument
log_level: Input should be 'debug', 'info' or 'warning'


Restrições também cabem numa dataclass stdlib, via `Annotated` (o Pydantic lê os metadados; para o Python, é só `int`):

In [7]:
from typing import Annotated
from pydantic import Field

Port = Annotated[int, Field(ge=1, le=65535)]


@dataclass(frozen=True)
class StrictServer:
    host: str
    port: Port


try:
    TypeAdapter(StrictServer).validate_python({"host": "x", "port": 70000})
except ValidationError as ex:
    print(f"port=70000: {ex.errors()[0]['msg']}")
print(f"StrictServer('x', 70000) direto (sem adapter): {StrictServer('x', 70000)!r}")

port=70000: Input should be less than or equal to 65535
StrictServer('x', 70000) direto (sem adapter): StrictServer(host='x', port=70000)


A última linha é o limite dessa abordagem: quem constrói a dataclass **sem** o adapter não recebe validação nenhuma.

## 4. `BaseModel` (ou `pydantic.dataclasses`): validação sempre ligada

In [8]:
from pydantic import BaseModel
from pydantic.dataclasses import dataclass as pydantic_dataclass


@pydantic_dataclass(frozen=True, config=ConfigDict(extra="forbid"))
class PdServer:
    host: str
    port: Port


try:
    PdServer("x", 70000)
except ValidationError as ex:
    print(f"PdServer('x', 70000): {ex.errors()[0]['msg']}")
print(f"PdServer('x', '8080'): {PdServer('x', '8080')!r}")

PdServer('x', 70000): Input should be less than or equal to 65535
PdServer('x', '8080'): PdServer(host='x', port=8080)


`pydantic.dataclasses.dataclass` mantém a "cara" de dataclass (`dataclasses.fields`, `asdict`, posicional) e valida no
`__init__`. `BaseModel` traz mais recursos (serialização, `model_validate_json`, aliases, campos calculados). Para configuração,
qualquer um dos dois serve; o que importa é a validação estar **sempre** ligada.

## 5. Camadas: base < ambiente < variáveis de ambiente < argumentos

Configuração real raramente vem de um arquivo só. O `pydantic-settings` combina **fontes** com precedência definida e faz merge
**aninhado** (só as chaves presentes na camada de cima são substituídas).

```
argumentos (init)          ← mais forte
variáveis de ambiente      APP_SERVER__PORT=...
configs/prod.yaml          só o que muda
configs/base.yaml          ← mais fraca
defaults do modelo
```

In [9]:
from pydantic import SecretStr
from pydantic_settings import BaseSettings, SettingsConfigDict, YamlConfigSettingsSource
from pydantic import BaseModel as PBaseModel


class AppS(PBaseModel):
    model_config = ConfigDict(extra="forbid")
    name: str
    environment: Environment


class ServerS(PBaseModel):
    model_config = ConfigDict(extra="forbid")
    host: str = "127.0.0.1"
    port: Port = 8080
    timeout_s: float = Field(default=5.0, gt=0)


class DatabaseS(PBaseModel):
    model_config = ConfigDict(extra="forbid")
    url: str
    pool_size: int = Field(default=5, ge=1)
    password: SecretStr = SecretStr("")


def settings_for(environment: str) -> type[BaseSettings]:
    files = [CONFIGS / "base.yaml"] + ([CONFIGS / f"{environment}.yaml"] if environment != "dev" else [])

    class AppSettings(BaseSettings):
        model_config = SettingsConfigDict(env_prefix="APP_", env_nested_delimiter="__", extra="forbid",
                                          yaml_file=files, frozen=True)
        app: AppS
        server: ServerS = ServerS()
        database: DatabaseS
        log_level: LogLevel = LogLevel.INFO
        features: tuple[str, ...] = ()

        @classmethod
        def settings_customise_sources(cls, settings_cls, init_settings, env_settings, dotenv_settings,
                                       file_secret_settings):
            return (init_settings, env_settings, YamlConfigSettingsSource(settings_cls, deep_merge=True))

    return AppSettings


dev = settings_for("dev")()
prod = settings_for("prod")()
print(f"dev.app.environment, dev.server.port, dev.database.pool_size: {dev.app.environment}, {dev.server.port}, {dev.database.pool_size}")
print(f"prod.app.environment, prod.server.port, prod.database.pool_size: {prod.app.environment}, {prod.server.port}, {prod.database.pool_size}")
print(f"prod.server.host (herdado da base): {prod.server.host}")

dev.app.environment, dev.server.port, dev.database.pool_size: dev, 8080, 5
prod.app.environment, prod.server.port, prod.database.pool_size: prod, 443, 20
prod.server.host (herdado da base): 0.0.0.0


Agora as variáveis de ambiente por cima (é assim que segredos e ajustes de implantação entram, sem tocar em arquivo):

In [10]:
os.environ["APP_SERVER__PORT"] = "8443"
os.environ["APP_DATABASE__PASSWORD"] = "s3cr3t"
try:
    prod = settings_for("prod")()
    print(f"prod.server.port (env vence o yaml): {prod.server.port}")
    print(f"prod.database.password: {prod.database.password!r}")
    print(f"prod.database.password.get_secret_value() == 's3cr3t': {prod.database.password.get_secret_value() == 's3cr3t'}")
    print(f"senha no model_dump_json: {'s3cr3t' in prod.model_dump_json()}")
    print(f"argumento vence tudo: {settings_for('prod')(log_level='debug').log_level!r}")
finally:
    del os.environ["APP_SERVER__PORT"], os.environ["APP_DATABASE__PASSWORD"]

prod.server.port (env vence o yaml): 8443
prod.database.password: SecretStr('**********')
prod.database.password.get_secret_value() == 's3cr3t': True
senha no model_dump_json: False
argumento vence tudo: <LogLevel.DEBUG: 'debug'>


E o erro de uma variável de ambiente mal escrita aparece com o mesmo caminho dos outros:

In [11]:
os.environ["APP_SERVER__PORT"] = "oitenta"
try:
    settings_for("prod")()
except ValidationError as ex:
    print(f"{'.'.join(map(str, ex.errors()[0]['loc']))}: {ex.errors()[0]['msg']} (veio {ex.errors()[0]['input']!r})")
finally:
    del os.environ["APP_SERVER__PORT"]

server.port: Input should be a valid integer, unable to parse string as an integer (veio 'oitenta')


> ⚠️ `extra="forbid"` em `BaseSettings` vale para **arquivos e argumentos**; variáveis de ambiente que não batem com nenhum campo
> são simplesmente ignoradas (o ambiente tem centenas de variáveis que não são suas). Um `APP_SERVR__PORT` com erro de digitação
> passa em silêncio — vale um teste que lista as variáveis `APP_*` e confere se todas são conhecidas.

In [12]:
def unknown_app_env_vars(settings_cls: type[BaseSettings], prefix: str = "APP_") -> list[str]:
    top = {name.upper() for name in settings_cls.model_fields}
    return [key for key in os.environ if key.startswith(prefix) and key[len(prefix):].split("__")[0] not in top]


os.environ["APP_SERVR__PORT"] = "9999"
try:
    print(f"settings carregou mesmo assim, porta: {settings_for('dev')().server.port}")
    print(f"variáveis APP_* desconhecidas: {unknown_app_env_vars(settings_for('dev'))}")
finally:
    del os.environ["APP_SERVR__PORT"]

settings carregou mesmo assim, porta: 8080
variáveis APP_* desconhecidas: ['APP_SERVR__PORT']


## 6. Enums na configuração: valor, nome e caixa

O Pydantic converte pelo **valor** do Enum. Arquivos escritos por humanos às vezes usam o **nome** (`WARNING`) ou outra caixa
(`Warning`). Três opções, da mais rígida à mais tolerante:

In [13]:
from pydantic import BeforeValidator


class Level(Enum):
    DEBUG = 10
    INFO = 20
    WARNING = 30


def by_name_or_value(value: Any) -> Any:
    if isinstance(value, str) and value.upper() in Level.__members__:
        return Level[value.upper()]
    return value


class LooseLevel(StrEnum):
    DEBUG = "debug"
    INFO = "info"

    @classmethod
    def _missing_(cls, value):
        if isinstance(value, str):
            return cls.__members__.get(value.upper())        # aceita "INFO", "Info"...
        return None


strict_level = TypeAdapter(Level)
named_level = TypeAdapter(Annotated[Level, BeforeValidator(by_name_or_value)])
for value in [30, "WARNING", "warning", "verbose"]:
    results = []
    for label, adapter in (("valor", strict_level), ("nome ou valor", named_level)):
        try:
            results.append(f"{label}={adapter.validate_python(value)!r}")
        except ValidationError:
            results.append(f"{label}=erro")
    print(f"{value!r}: {', '.join(results)}")
print(f"LooseLevel('INFO') via _missing_: {TypeAdapter(LooseLevel).validate_python('INFO')!r}")

30: valor=<Level.WARNING: 30>, nome ou valor=<Level.WARNING: 30>
'WARNING': valor=erro, nome ou valor=<Level.WARNING: 30>
'warning': valor=erro, nome ou valor=<Level.WARNING: 30>
'verbose': valor=erro, nome ou valor=erro
LooseLevel('INFO') via _missing_: <LooseLevel.INFO: 'info'>


`_missing_` muda o Enum **para todo mundo** (inclusive código que não é de configuração); o `BeforeValidator` muda só a leitura da
configuração. Prefira o validador.

## 7. Congelar depois de carregar

Configuração carregada não deve mudar em runtime. `frozen=True` em `AppSettings` impede trocar os atributos **dele**:

In [14]:
frozen_check = settings_for("prod")()
try:
    frozen_check.log_level = LogLevel.DEBUG
except ValidationError as ex:
    print(f"frozen_check.log_level = ...: {ex.errors()[0]['type']}")

frozen_check.log_level = ...: frozen_instance


Mas o congelamento é **raso**: `ServerS` não é `frozen`, então o submodelo aceita a atribuição sem reclamar:

In [15]:
frozen_check.server.port = 1
print(f"frozen_check.server.port: {frozen_check.server.port}")

frozen_check.server.port: 1


Regra prática: congele **cada** modelo de configuração (`frozen=True` em todos) e use `tuple`/`frozenset` no lugar de `list`/`set`
(aqui `features` já é `tuple`). Com o submodelo congelado:

In [16]:
class FrozenServer(PBaseModel):
    model_config = ConfigDict(extra="forbid", frozen=True)
    host: str = "127.0.0.1"
    port: Port = 8080


server = FrozenServer()
try:
    server.port = 1
except ValidationError as ex:
    print(f"server.port = 1: {ex.errors()[0]['type']}")
print(f"type(frozen_check.features): {type(frozen_check.features).__name__}")

server.port = 1: frozen_instance
type(frozen_check.features): tuple


## ⚠️ Pegadinhas, efeitos colaterais e caveats

### Já demonstradas acima

| # | pegadinha | onde | como evitar |
|---|---|---|---|
| 1 | `Dataclass(**dict)` não converte nada; anotações são só documentação | §1 | validar na fronteira (`TypeAdapter`) |
| 2 | conversor caseiro: `int(True)`, `int("8080.7")`, sem caminho do erro, para no primeiro | §2 | não reinventar; usar o Pydantic |
| 3 | `TypeAdapter(dataclass)` **ignora** chaves extras por padrão | §3 | `__pydantic_config__ = {"extra": "forbid"}` na classe de fora |
| 4 | `TypeAdapter(..., config=...)` é recusado para dataclasses | §3 | `__pydantic_config__` |
| 5 | dataclass construída sem o adapter não valida nada | §3 | `pydantic.dataclasses` ou validar sempre na borda |
| 6 | `extra="forbid"` não pega `APP_SERVR__PORT` (variável de ambiente com erro de digitação) | §5 | conferir as variáveis com o prefixo |
| 7 | `_missing_` muda o Enum para todo o programa | §6 | `BeforeValidator` só na leitura da configuração |
| 8 | `frozen` é raso | §7 | `frozen` em todos os níveis + `tuple` |

### Novas

**1. Lista em variável de ambiente precisa ser JSON.**

In [17]:
from pydantic_settings import SettingsError


class FeatureSettings(BaseSettings):
    model_config = SettingsConfigDict(env_prefix="DEMO_")
    features: list[str] = []
    debug: bool = False


os.environ["DEMO_FEATURES"] = "export_csv,export_pdf"
try:
    FeatureSettings()
except SettingsError as ex:
    print(f"'export_csv,export_pdf': {ex}")
os.environ["DEMO_FEATURES"] = '["export_csv", "export_pdf"]'
print(f"JSON: {FeatureSettings().features}")
del os.environ["DEMO_FEATURES"]

'export_csv,export_pdf': error parsing value for field "features" from source "EnvSettingsSource"
JSON: ['export_csv', 'export_pdf']


**2. Booleano vindo do ambiente: `"off"` e `"0"` são `False`; vazio é erro.**

In [18]:
for raw_value in ["off", "0", "no", ""]:
    os.environ["DEMO_DEBUG"] = raw_value
    try:
        print(f"DEMO_DEBUG={raw_value!r}: {FeatureSettings().debug}")
    except ValidationError as ex:
        print(f"DEMO_DEBUG={raw_value!r}: {ex.errors()[0]['msg']}")
del os.environ["DEMO_DEBUG"]

DEMO_DEBUG='off': False
DEMO_DEBUG='0': False
DEMO_DEBUG='no': False
DEMO_DEBUG='': Input should be a valid boolean, unable to interpret input


`export DEMO_DEBUG=` (comum em scripts que "limpam" uma variável) derruba a aplicação na carga. Para desligar, **remova** a variável
(`unset`) ou use `0`.

**3. Segredo mascarado + ida e volta = segredo destruído.** O `SecretStr` mascara o valor no `repr` e no JSON. Ótimo para logs;
perigoso se alguém usa o dump para **salvar** a configuração e depois recarregá-la:

In [19]:
os.environ["APP_DATABASE__PASSWORD"] = "s3cr3t"
try:
    original = settings_for("prod")()
finally:
    del os.environ["APP_DATABASE__PASSWORD"]
saved = original.model_dump(mode="json")                     # "salvar a configuração efetiva"
reloaded = settings_for("prod")(**saved)
print(f"saved['database']['password']: {saved['database']['password']}")
print(f"senha depois de recarregar: {reloaded.database.password.get_secret_value()!r}")

saved['database']['password']: **********
senha depois de recarregar: '**********'


A senha passou a ser **literalmente** `**********`, sem erro nenhum — e a falha só aparece quando o banco recusar a conexão.
Dumps de configuração servem para auditoria, nunca como fonte para recarregar; segredos vêm sempre do cofre/ambiente.

**4. `@lru_cache` em `get_settings()`: a configuração congela no primeiro acesso.** É o padrão recomendado em várias
documentações (evita reler arquivos), com um efeito colateral em testes e em processos longos:

In [20]:
from functools import lru_cache


@lru_cache
def get_settings():
    return settings_for("dev")()


first = get_settings()
os.environ["APP_SERVER__PORT"] = "9999"
try:
    print(f"porta depois de mudar o ambiente: {get_settings().server.port}")
    get_settings.cache_clear()
    print(f"depois de cache_clear(): {get_settings().server.port}")
finally:
    del os.environ["APP_SERVER__PORT"]
    get_settings.cache_clear()

porta depois de mudar o ambiente: 8080
depois de cache_clear(): 9999


Em testes, um `cache_clear()` em *fixture* evita que um teste herde a configuração do anterior. O mesmo vale para settings
instanciados no **import** de um módulo: congelam o ambiente daquele momento.

**5. Caminhos relativos na configuração são relativos ao diretório de trabalho, não ao arquivo.**

In [21]:
config_file = CONFIGS / "base.yaml"
relative = Path("dados/vendas.csv")                               # como estaria escrito no YAML
print(f"relativo ao cwd: {relative.resolve().relative_to(Path.cwd())}")
print(f"relativo ao arquivo: {(config_file.parent / relative).resolve().relative_to(Path.cwd())}")

relativo ao cwd: dados/vendas.csv
relativo ao arquivo: configs/dados/vendas.csv


Rodar a mesma aplicação de outra pasta (um `cron`, um container com `WORKDIR` diferente) faz o caminho apontar para outro lugar.
Resolva caminhos relativos contra a pasta do **arquivo de configuração** logo na carga (um validador que recebe a pasta via
`context`, ou um passo de pós-processamento) — ou exija caminhos absolutos.

**6. Testes que mexem em `os.environ` vazam para os seguintes.** Todos os exemplos deste notebook usam `try/finally` para remover
as variáveis. Em `pytest`, a *fixture* `monkeypatch.setenv` faz isso sozinha. Conferindo que nada ficou para trás:

In [22]:
print(f"variáveis APP_*/DEMO_* restantes: {[k for k in os.environ if k.startswith(('APP_', 'DEMO_'))]}")

variáveis APP_*/DEMO_* restantes: []


## 8. Resumo

```mermaid
flowchart LR
    F1[base.yaml] --> M{merge aninhado}
    F2[prod.yaml] --> M
    E[env APP_*] --> M
    A[argumentos] --> M
    M --> V[validação Pydantic<br/>tipos · restrições · extras · enums]
    V -->|ok| C[objeto congelado]
    V -->|erro| X[ValidationError com caminho<br/>server.port: ...]
```

- Dataclass pura **não** é camada de validação; use-a para o domínio e valide na fronteira (`TypeAdapter`).
- Erros precisam do **caminho** (`server.port`) — é o que torna a mensagem útil para quem editou o arquivo.
- Camadas: arquivo base + arquivo por ambiente (só diferenças) + env para segredos/implantação + argumentos.
- Congele **todos** os níveis; `extra="forbid"` não cobre variáveis de ambiente.